# morphological_quantification_2026-01-02 — 05g_transverse_distance_profiles

**Feeds:** Fig 3f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 05g | Transverse Distance Profiles For Marker Positivity

This notebook summarizes where `MESP2`, `PAX8`, and `FOXF1` positive pixels sit relative to the
posterior-oriented consensus axis. For each organoid, we bin **all** whole-morph mask pixels by
their absolute transverse distance from the axis, then compute the fraction of those pixels that
are marker-positive inside each bin.

The plot shows:
- thin per-organoid traces (one per marker)
- thick cohort mean traces
- shaded ±1 SD across organoids


## Cell Guide

- `Setup`: resolve the project root, import helper code, and define bins.
- `Load Inputs`: read the consensus axis table, per-z whole-morph masks, and marker-positive masks.
- `Compute Per-Organoid Profiles`: bin transverse distances and compute positive fractions.
- `Save Summary Tables`: per-organoid profiles plus cohort mean/SD.
- `Plot`: thin organoid traces plus mean ± SD in shared axes.


In [ ]:
import json
import sys
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    ROOT = cwd.parent
elif cwd.name == "executed_notebooks" and cwd.parent.name == "results":
    ROOT = cwd.parent.parent
elif (cwd / "notebooks").exists():
    ROOT = cwd
elif cwd.parent.name == "results" and (cwd.parent.parent / "notebooks").exists():
    ROOT = cwd.parent.parent
else:
    raise RuntimeError(
        "Run this notebook from the project root, notebooks/, or results/executed_notebooks/."
    )

SCRIPTS_DIR = ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import morphology_quantification_helpers as mqh

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 120)


## Settings Notes

- Transverse distance is absolute (medial-lateral sign is ignored).
- Bins are fixed-width in `um` and shared across all organoids.
- For each bin, the denominator is **all** mask pixels in that bin; the numerator is the number
  of marker-positive pixels in the same bin.
- We do not apply a minimum-pixel cutoff per bin in this version.


In [ ]:
AXIS_TABLE_PATH = ROOT / "results" / "tables" / "05_posterior_oriented_consensus_axes.tsv"
GEOMETRY_TABLE_PATH = ROOT / "results" / "tables" / "whole_morph_per_z_geometry.tsv"
DOMAIN_PLANE_TABLE_PATH = ROOT / "results" / "tables" / "05_marker_domain_metrics_by_plane.tsv"
ANALYSIS_MANIFEST_PATH = ROOT / "results" / "manifests" / "analysis_manifest.tsv"
PARAMETERS_PATH = ROOT / "results" / "tables" / "04_marker_positive_region_parameters.json"

OUTPUT_TABLE_PATH = ROOT / "results" / "tables" / "05g_transverse_distance_profiles_by_file.tsv"
SUMMARY_TABLE_PATH = ROOT / "results" / "tables" / "05g_transverse_distance_profiles_summary.tsv"
OUTPUT_QC_DIR = ROOT / "results" / "qc" / "transverse_distance_profiles"
OUTPUT_QC_DIR.mkdir(parents=True, exist_ok=True)
PLOT_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles.png"
PLOT_PATH_CI = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_ci.png"
PLOT_FRACTION_NORM_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm.png"
PLOT_FRACTION_NORM_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm_ci.png"
PLOT_FRACTION_MINMAX_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax.png"
PLOT_FRACTION_MINMAX_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax_ci.png"
PLOT_FRACTION_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_mean_only.png"
PLOT_FRACTION_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_mean_only_ci.png"
PLOT_FRACTION_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm_mean_only.png"
PLOT_FRACTION_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm_mean_only_ci.png"
PLOT_FRACTION_MINMAX_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax_mean_only.png"
PLOT_FRACTION_MINMAX_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax_mean_only_ci.png"
PLOT_INTENSITY_RAW_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw.png"
PLOT_INTENSITY_RAW_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw_ci.png"
PLOT_INTENSITY_NORM_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm.png"
PLOT_INTENSITY_NORM_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm_ci.png"
PLOT_INTENSITY_ALL_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all.png"
PLOT_INTENSITY_ALL_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all_ci.png"
PLOT_INTENSITY_RAW_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw_mean_only.png"
PLOT_INTENSITY_RAW_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw_mean_only_ci.png"
PLOT_INTENSITY_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm_mean_only.png"
PLOT_INTENSITY_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm_mean_only_ci.png"
PLOT_INTENSITY_ALL_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all_mean_only.png"
PLOT_INTENSITY_ALL_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all_mean_only_ci.png"
PLOT_SOFT_RAW_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw.png"
PLOT_SOFT_RAW_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw_ci.png"
PLOT_SOFT_NORM_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm.png"
PLOT_SOFT_NORM_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm_ci.png"
PLOT_SOFT_RAW_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw_mean_only.png"
PLOT_SOFT_RAW_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw_mean_only_ci.png"
PLOT_SOFT_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm_mean_only.png"
PLOT_SOFT_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm_mean_only_ci.png"
PLOT_FRACTION_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_mean_only.png"
PLOT_FRACTION_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_mean_only_ci.png"
PLOT_FRACTION_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm_mean_only.png"
PLOT_FRACTION_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm_mean_only_ci.png"
PLOT_FRACTION_MINMAX_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax_mean_only.png"
PLOT_FRACTION_MINMAX_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax_mean_only_ci.png"
PLOT_INTENSITY_RAW_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw_mean_only.png"
PLOT_INTENSITY_RAW_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw_mean_only_ci.png"
PLOT_INTENSITY_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm_mean_only.png"
PLOT_INTENSITY_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm_mean_only_ci.png"
PLOT_INTENSITY_ALL_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all_mean_only.png"
PLOT_INTENSITY_ALL_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all_mean_only_ci.png"
PLOT_SOFT_RAW_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw_mean_only.png"
PLOT_SOFT_RAW_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw_mean_only_ci.png"
PLOT_SOFT_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm_mean_only.png"
PLOT_SOFT_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm_mean_only_ci.png"
PLOT_FRACTION_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_mean_only.png"
PLOT_FRACTION_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_mean_only_ci.png"
PLOT_FRACTION_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm_mean_only.png"
PLOT_FRACTION_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm_mean_only_ci.png"
PLOT_FRACTION_MINMAX_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax_mean_only.png"
PLOT_FRACTION_MINMAX_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax_mean_only_ci.png"
PLOT_INTENSITY_RAW_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw_mean_only.png"
PLOT_INTENSITY_RAW_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw_mean_only_ci.png"
PLOT_INTENSITY_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm_mean_only.png"
PLOT_INTENSITY_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm_mean_only_ci.png"
PLOT_INTENSITY_ALL_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all_mean_only.png"
PLOT_INTENSITY_ALL_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all_mean_only_ci.png"
PLOT_SOFT_RAW_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw_mean_only.png"
PLOT_SOFT_RAW_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw_mean_only_ci.png"
PLOT_SOFT_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm_mean_only.png"
PLOT_SOFT_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm_mean_only_ci.png"
PLOT_FRACTION_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_mean_only.png"
PLOT_FRACTION_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_mean_only_ci.png"
PLOT_FRACTION_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm_mean_only.png"
PLOT_FRACTION_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_norm_mean_only_ci.png"
PLOT_FRACTION_MINMAX_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax_mean_only.png"
PLOT_FRACTION_MINMAX_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_fraction_minmax_mean_only_ci.png"
PLOT_INTENSITY_RAW_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw_mean_only.png"
PLOT_INTENSITY_RAW_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_raw_mean_only_ci.png"
PLOT_INTENSITY_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm_mean_only.png"
PLOT_INTENSITY_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_norm_mean_only_ci.png"
PLOT_INTENSITY_ALL_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all_mean_only.png"
PLOT_INTENSITY_ALL_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_intensity_all_mean_only_ci.png"
PLOT_SOFT_RAW_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw_mean_only.png"
PLOT_SOFT_RAW_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_raw_mean_only_ci.png"
PLOT_SOFT_NORM_MEAN_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm_mean_only.png"
PLOT_SOFT_NORM_MEAN_CI_PATH = OUTPUT_QC_DIR / "05g_transverse_distance_profiles_soft_threshold_norm_mean_only_ci.png"

MARKERS = ["mesp2", "foxf1", "pax8"]
COLOR_MAP = {"mesp2": "#ef4444", "pax8": "#facc15", "foxf1": "#22d3ee"}
N_BINS = 24


## Load Inputs


In [ ]:
manifest_df = pd.read_csv(ANALYSIS_MANIFEST_PATH, sep="\t")
included_paths = set(
    manifest_df.loc[
        manifest_df["include_in_analysis"].fillna(True).astype(bool),
        "file_path",
    ].astype(str)
)

axis_df = pd.read_csv(AXIS_TABLE_PATH, sep="\t")
axis_df = axis_df.loc[axis_df["file_path"].astype(str).isin(included_paths)].copy()
axis_df["file_path"] = axis_df["file_path"].astype(str)

geom_df = pd.read_csv(GEOMETRY_TABLE_PATH, sep="\t")
geom_df = geom_df.loc[geom_df["file_path"].astype(str).isin(included_paths)].copy()
geom_df["file_path"] = geom_df["file_path"].astype(str)

domain_plane_df = pd.read_csv(DOMAIN_PLANE_TABLE_PATH, sep="\t")
domain_plane_df = domain_plane_df.loc[
    domain_plane_df["file_path"].astype(str).isin(included_paths)
    & domain_plane_df["marker_key"].astype(str).isin(MARKERS)
].copy()

if not PARAMETERS_PATH.exists():
    raise RuntimeError(f"Missing parameters file: {PARAMETERS_PATH}")
parameters = json.loads(PARAMETERS_PATH.read_text())
BACKGROUND_ESTIMATOR = str(parameters.get("background_estimator", "whole_off_morph"))
ANNULUS_INNER_RADIUS_PX = int(parameters.get("annulus_inner_radius_px", 6))
ANNULUS_OUTER_RADIUS_PX = int(parameters.get("annulus_outer_radius_px", 20))
MIN_RING_PIXELS = int(parameters.get("min_ring_pixels", 2000))


## Prepare Axis Cache And Mask Lookup


In [ ]:
axis_cache = {}
for row in axis_df.itertuples(index=False):
    axis_cache[str(row.file_path)] = {
        "file_id": int(row.file_id),
        "centerline_xy": np.asarray(json.loads(row.centerline_xy_json), dtype=np.float64),
        "pixel_size_um": float(row.pixel_size_um),
        "retained_z_indices": set(mqh.parse_int_list_field(getattr(row, "retained_z_indices", ""))),
    }

if not axis_cache:
    raise RuntimeError("No axis rows found for included files.")

@lru_cache(maxsize=4096)
def load_binary_mask_cached(rel_path: str) -> np.ndarray:
    return mqh.load_binary_mask(ROOT / str(rel_path))

# Map positive-mask paths for quick lookup: (file_path, z_index, marker_key) -> mask_path
pos_mask_lookup = {}
threshold_lookup = {}
for row in domain_plane_df.itertuples(index=False):
    key = (str(row.file_path), int(row.z_index), str(row.marker_key))
    pos_mask_lookup[key] = str(row.positive_mask_path)
    threshold_lookup[key] = float(row.threshold_value)


## Determine Shared Distance Bins


In [ ]:
def max_abs_transverse_um_for_file(file_path: str) -> float:
    axis_info = axis_cache.get(file_path)
    if axis_info is None:
        return 0.0
    centerline_xy = axis_info["centerline_xy"]
    pixel_size_um = float(axis_info["pixel_size_um"])
    retained = axis_info["retained_z_indices"]
    if not retained:
        retained = set(
            geom_df.loc[geom_df["file_path"].astype(str) == file_path, "z_index"]
            .astype(int)
            .tolist()
        )
    max_um = 0.0
    file_geom = geom_df.loc[geom_df["file_path"].astype(str) == file_path]
    for row in file_geom.itertuples(index=False):
        if int(row.z_index) not in retained:
            continue
        mask = load_binary_mask_cached(str(row.mask_path))
        ys, xs = np.where(mask)
        if len(xs) == 0:
            continue
        pts = np.column_stack([xs, ys]).astype(np.float64)
        proj = mqh.project_points_to_centerline(
            pts,
            centerline_xy,
            n_samples=401,
        )
        abs_px = np.asarray(proj["transverse_abs_px"], dtype=np.float64)
        if abs_px.size:
            max_um = max(max_um, float(np.nanmax(abs_px) * pixel_size_um))
    return float(max_um)

max_um_values = [max_abs_transverse_um_for_file(fp) for fp in sorted(axis_cache)]
global_max_um = float(np.nanmax(max_um_values)) if max_um_values else 0.0
if not np.isfinite(global_max_um) or global_max_um <= 0:
    raise RuntimeError("Unable to determine transverse-distance range for bins.")

bin_edges = np.linspace(0.0, global_max_um, int(N_BINS) + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])


## Compute Per-Organoid Profiles


In [ ]:
rows = []

for file_path, axis_info in axis_cache.items():
    file_id = int(axis_info["file_id"])
    centerline_xy = axis_info["centerline_xy"]
    pixel_size_um = float(axis_info["pixel_size_um"])
    retained = axis_info["retained_z_indices"]
    if not retained:
        retained = set(
            geom_df.loc[geom_df["file_path"].astype(str) == file_path, "z_index"]
            .astype(int)
            .tolist()
        )

    stack = mqh.load_czi_stack(mqh._resolve_local_path(file_path, ROOT))
    channel_idx_map = mqh.marker_channel_index_map(stack.channels)
    marker_channels = {marker_key: channel_idx_map.get(marker_key) for marker_key in MARKERS}

    mask_counts = np.zeros(len(bin_centers), dtype=np.int64)
    pos_counts = {m: np.zeros(len(bin_centers), dtype=np.int64) for m in MARKERS}
    intensity_sums = {m: np.zeros(len(bin_centers), dtype=np.float64) for m in MARKERS}
    intensity_all_sums = {m: np.zeros(len(bin_centers), dtype=np.float64) for m in MARKERS}
    soft_sums = {m: np.zeros(len(bin_centers), dtype=np.float64) for m in MARKERS}

    file_geom = geom_df.loc[geom_df["file_path"].astype(str) == file_path]
    for row in file_geom.itertuples(index=False):
        z_index = int(row.z_index)
        if z_index not in retained:
            continue
        organoid_mask = load_binary_mask_cached(str(row.mask_path))
        ys, xs = np.where(organoid_mask)
        if len(xs) == 0:
            continue
        pts = np.column_stack([xs, ys]).astype(np.float64)
        proj = mqh.project_points_to_centerline(
            pts,
            centerline_xy,
            n_samples=401,
        )
        abs_um = np.asarray(proj["transverse_abs_px"], dtype=np.float64) * pixel_size_um
        mask_counts += np.histogram(abs_um, bins=bin_edges)[0].astype(np.int64)

        for marker_key in MARKERS:
            ch_idx = marker_channels.get(marker_key)
            if ch_idx is None:
                continue
            key = (file_path, z_index, marker_key)
            threshold_value = threshold_lookup.get(key)
            if threshold_value is None:
                continue
            signal = np.asarray(stack.data_czyx[int(ch_idx), z_index], dtype=np.float32)
            corrected, _, _ = mqh.background_correct_signal(
                signal=signal,
                organoid_mask=organoid_mask,
                background_estimator=BACKGROUND_ESTIMATOR,
                annulus_inner_radius_px=ANNULUS_INNER_RADIUS_PX,
                annulus_outer_radius_px=ANNULUS_OUTER_RADIUS_PX,
                min_ring_pixels=MIN_RING_PIXELS,
            )
            mask_vals = corrected[ys, xs].astype(np.float64)
            finite_mask_all = np.isfinite(mask_vals)
            if np.any(finite_mask_all):
                mask_vals = mask_vals[finite_mask_all]
                abs_um_all = abs_um[finite_mask_all]
                bin_idx_all = np.digitize(abs_um_all, bin_edges) - 1
                valid_all = (bin_idx_all >= 0) & (bin_idx_all < len(bin_centers))
                if np.any(valid_all):
                    intensity_all_sums[marker_key] += np.bincount(
                        bin_idx_all[valid_all],
                        weights=mask_vals[valid_all],
                        minlength=len(bin_centers),
                    )
                    soft_vals = np.maximum(mask_vals[valid_all] - float(threshold_value), 0.0)
                    soft_sums[marker_key] += np.bincount(
                        bin_idx_all[valid_all],
                        weights=soft_vals,
                        minlength=len(bin_centers),
                    )

            mask_path = pos_mask_lookup.get(key)
            if mask_path is None:
                continue
            pos_mask = load_binary_mask_cached(mask_path)
            pos_ys, pos_xs = np.where(pos_mask)
            if len(pos_xs) == 0:
                continue
            pos_pts = np.column_stack([pos_xs, pos_ys]).astype(np.float64)
            pos_proj = mqh.project_points_to_centerline(
                pos_pts,
                centerline_xy,
                n_samples=401,
            )
            pos_abs_um = np.asarray(pos_proj["transverse_abs_px"], dtype=np.float64) * pixel_size_um
            pos_counts[marker_key] += np.histogram(pos_abs_um, bins=bin_edges)[0].astype(np.int64)

            pos_vals = corrected[pos_ys, pos_xs].astype(np.float64)
            finite_mask = np.isfinite(pos_vals)
            if not np.any(finite_mask):
                continue
            pos_vals = pos_vals[finite_mask]
            pos_abs_um_valid = pos_abs_um[finite_mask]
            bin_idx = np.digitize(pos_abs_um_valid, bin_edges) - 1
            valid = (bin_idx >= 0) & (bin_idx < len(bin_centers))
            if not np.any(valid):
                continue
            intensity_sums[marker_key] += np.bincount(
                bin_idx[valid],
                weights=pos_vals[valid],
                minlength=len(bin_centers),
            )

    total_intensity = {
        marker_key: float(np.nansum(intensity_sums[marker_key]))
        for marker_key in MARKERS
    }
    total_intensity_all = {
        marker_key: float(np.nansum(intensity_all_sums[marker_key]))
        for marker_key in MARKERS
    }
    total_soft = {
        marker_key: float(np.nansum(soft_sums[marker_key]))
        for marker_key in MARKERS
    }

    for idx, center_um in enumerate(bin_centers):
        denom = float(mask_counts[idx])
        for marker_key in MARKERS:
            numer = float(pos_counts[marker_key][idx])
            frac = numer / denom if denom > 0 else np.nan
            intensity_numer = float(intensity_sums[marker_key][idx])
            intensity_total = total_intensity.get(marker_key, 0.0)
            intensity_frac = (
                intensity_numer / intensity_total
                if intensity_total > 0
                else np.nan
            )
            intensity_all_numer = float(intensity_all_sums[marker_key][idx])
            intensity_all_total = total_intensity_all.get(marker_key, 0.0)
            intensity_all_frac = (
                intensity_all_numer / intensity_all_total
                if intensity_all_total > 0
                else np.nan
            )
            soft_numer = float(soft_sums[marker_key][idx])
            soft_total = total_soft.get(marker_key, 0.0)
            soft_frac = soft_numer / soft_total if soft_total > 0 else np.nan
            rows.append(
                {
                    "file_id": file_id,
                    "file_path": file_path,
                    "marker_key": marker_key,
                    "bin_index": int(idx),
                    "bin_center_um": float(center_um),
                    "mask_pixel_count": int(mask_counts[idx]),
                    "positive_pixel_count": int(pos_counts[marker_key][idx]),
                    "positive_fraction": float(frac) if np.isfinite(frac) else np.nan,
                    "positive_intensity_sum": float(intensity_numer),
                    "positive_intensity_fraction": float(intensity_frac)
                    if np.isfinite(intensity_frac)
                    else np.nan,
                    "intensity_sum": float(intensity_all_numer),
                    "intensity_fraction": float(intensity_all_frac)
                    if np.isfinite(intensity_all_frac)
                    else np.nan,
                    "soft_intensity_sum": float(soft_numer),
                    "soft_intensity_fraction": float(soft_frac) if np.isfinite(soft_frac) else np.nan,
                }
            )

profile_df = pd.DataFrame(rows)

if profile_df.empty:
    raise RuntimeError("No transverse-distance profile rows were computed.")

profile_df.to_csv(OUTPUT_TABLE_PATH, sep="\t", index=False)


## Cohort Mean / SD


In [ ]:
summary_rows = []
for marker_key in MARKERS:
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key].copy()
    for bin_idx, bin_center_um in enumerate(bin_centers):
        bin_sub = sub.loc[sub["bin_index"].astype(int) == int(bin_idx)]
        values = pd.to_numeric(bin_sub["positive_fraction"], errors="coerce")
        values_intensity = pd.to_numeric(bin_sub["positive_intensity_fraction"], errors="coerce")
        values_intensity_all = pd.to_numeric(bin_sub["intensity_fraction"], errors="coerce")
        values_soft = pd.to_numeric(bin_sub["soft_intensity_fraction"], errors="coerce")
        mean_val = float(np.nanmean(values)) if values.notna().any() else np.nan
        std_val = float(np.nanstd(values)) if values.notna().any() else np.nan
        mean_intensity = float(np.nanmean(values_intensity)) if values_intensity.notna().any() else np.nan
        std_intensity = float(np.nanstd(values_intensity)) if values_intensity.notna().any() else np.nan
        mean_intensity_all = (
            float(np.nanmean(values_intensity_all)) if values_intensity_all.notna().any() else np.nan
        )
        std_intensity_all = (
            float(np.nanstd(values_intensity_all)) if values_intensity_all.notna().any() else np.nan
        )
        mean_soft = float(np.nanmean(values_soft)) if values_soft.notna().any() else np.nan
        std_soft = float(np.nanstd(values_soft)) if values_soft.notna().any() else np.nan
        summary_rows.append(
            {
                "marker_key": marker_key,
                "bin_index": int(bin_idx),
                "bin_center_um": float(bin_center_um),
                "mean_fraction": mean_val,
                "std_fraction": std_val,
                "mean_intensity_fraction": mean_intensity,
                "std_intensity_fraction": std_intensity,
                "mean_intensity_all_fraction": mean_intensity_all,
                "std_intensity_all_fraction": std_intensity_all,
                "mean_soft_intensity_fraction": mean_soft,
                "std_soft_intensity_fraction": std_soft,
                "n_organoids": int(values.notna().sum()),
            }
        )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_TABLE_PATH, sep="\t", index=False)


## Plot (Positive Fraction)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_fraction"].to_numpy(dtype=float)))
max_mean = float(np.nanmax(summary_df["mean_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["positive_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_fraction"].to_numpy(dtype=float)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of pixels that are marker-positive")
ax.set_title("Marker positivity vs transverse distance (fixed bins)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote profile plot:", PLOT_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Fraction, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_fraction"].to_numpy(dtype=float)))
max_mean = float(np.nanmax(summary_df["mean_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["positive_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of pixels that are marker-positive")
ax.set_title("Marker positivity vs transverse distance (95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_PATH_CI, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_PATH_CI.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote profile plot (CI):", PLOT_PATH_CI.relative_to(ROOT).as_posix())


## Plot (Positive Fraction, Mean Peak Normalized)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0
max_mean = 1.0

scale_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    max_mean = float(np.nanmax(summary_sub["mean_fraction"].to_numpy(dtype=float)))
    scale_by_marker[marker_key] = max_mean if np.isfinite(max_mean) and max_mean > 0 else np.nan

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    scale = scale_by_marker.get(marker_key, np.nan)
    for file_id, g in sub.groupby("file_id"):
        y = g["positive_fraction"].to_numpy(dtype=float)
        if np.isfinite(scale) and scale > 0:
            y = y / scale
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_fraction"].to_numpy(dtype=float)
    if np.isfinite(scale) and scale > 0:
        mean = mean / scale
        std = std / scale
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Positive fraction (mean peak normalized to 1)")
ax.set_title("Marker positivity vs transverse distance (normalized)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_FRACTION_NORM_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_FRACTION_NORM_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote normalized fraction plot:", PLOT_FRACTION_NORM_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Fraction, Mean Peak Normalized, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0
max_mean = 1.0

scale_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    max_mean = float(np.nanmax(summary_sub["mean_fraction"].to_numpy(dtype=float)))
    scale_by_marker[marker_key] = max_mean if np.isfinite(max_mean) and max_mean > 0 else np.nan

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    scale = scale_by_marker.get(marker_key, np.nan)
    for file_id, g in sub.groupby("file_id"):
        y = g["positive_fraction"].to_numpy(dtype=float)
        if np.isfinite(scale) and scale > 0:
            y = y / scale
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    if np.isfinite(scale) and scale > 0:
        mean = mean / scale
        std = std / scale
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Positive fraction (mean peak normalized to 1)")
ax.set_title("Marker positivity vs transverse distance (normalized, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_FRACTION_NORM_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_FRACTION_NORM_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote normalized fraction plot (CI):", PLOT_FRACTION_NORM_CI_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Fraction, Mean Trace Min-Max)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0
max_mean = 1.0

minmax_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    mean_vals = summary_sub["mean_fraction"].to_numpy(dtype=float)
    min_val = float(np.nanmin(mean_vals))
    max_val = float(np.nanmax(mean_vals))
    minmax_by_marker[marker_key] = (min_val, max_val)

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    min_val, max_val = minmax_by_marker.get(marker_key, (np.nan, np.nan))
    denom = max_val - min_val if np.isfinite(max_val) and np.isfinite(min_val) else np.nan
    for file_id, g in sub.groupby("file_id"):
        y = g["positive_fraction"].to_numpy(dtype=float)
        if np.isfinite(denom) and denom > 0:
            y = (y - min_val) / denom
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_fraction"].to_numpy(dtype=float)
    if np.isfinite(denom) and denom > 0:
        mean = (mean - min_val) / denom
        std = std / denom
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Positive fraction (mean trace min-max to [0,1])")
ax.set_title("Marker positivity vs transverse distance (min-max)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_FRACTION_MINMAX_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_FRACTION_MINMAX_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote min-max fraction plot:", PLOT_FRACTION_MINMAX_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Fraction, Mean Trace Min-Max, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0
max_mean = 1.0

minmax_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    mean_vals = summary_sub["mean_fraction"].to_numpy(dtype=float)
    min_val = float(np.nanmin(mean_vals))
    max_val = float(np.nanmax(mean_vals))
    minmax_by_marker[marker_key] = (min_val, max_val)

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    min_val, max_val = minmax_by_marker.get(marker_key, (np.nan, np.nan))
    denom = max_val - min_val if np.isfinite(max_val) and np.isfinite(min_val) else np.nan
    for file_id, g in sub.groupby("file_id"):
        y = g["positive_fraction"].to_numpy(dtype=float)
        if np.isfinite(denom) and denom > 0:
            y = (y - min_val) / denom
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    if np.isfinite(denom) and denom > 0:
        mean = (mean - min_val) / denom
        std = std / denom
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Positive fraction (mean trace min-max to [0,1])")
ax.set_title("Marker positivity vs transverse distance (min-max, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_FRACTION_MINMAX_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_FRACTION_MINMAX_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote min-max fraction plot (CI):", PLOT_FRACTION_MINMAX_CI_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Intensity, Raw)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_intensity_fraction"].to_numpy(dtype=float)))
max_mean = float(np.nanmax(summary_df["mean_intensity_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["positive_intensity_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_intensity_fraction"].to_numpy(dtype=float)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of positive intensity in bin")
ax.set_title("Marker-positive intensity vs transverse distance (fixed bins)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_INTENSITY_RAW_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_INTENSITY_RAW_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote intensity profile plot:", PLOT_INTENSITY_RAW_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Intensity, Raw, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_intensity_fraction"].to_numpy(dtype=float)))
max_mean = float(np.nanmax(summary_df["mean_intensity_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["positive_intensity_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_intensity_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of positive intensity in bin")
ax.set_title("Marker-positive intensity vs transverse distance (95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_INTENSITY_RAW_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_INTENSITY_RAW_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote intensity profile plot (CI):", PLOT_INTENSITY_RAW_CI_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Intensity, Mean Peak Normalized)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0
max_mean = 1.0

scale_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    max_mean = float(np.nanmax(summary_sub["mean_intensity_fraction"].to_numpy(dtype=float)))
    scale_by_marker[marker_key] = max_mean if np.isfinite(max_mean) and max_mean > 0 else np.nan

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    scale = scale_by_marker.get(marker_key, np.nan)
    for file_id, g in sub.groupby("file_id"):
        y = g["positive_intensity_fraction"].to_numpy(dtype=float)
        if np.isfinite(scale) and scale > 0:
            y = y / scale
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_intensity_fraction"].to_numpy(dtype=float)
    if np.isfinite(scale) and scale > 0:
        mean = mean / scale
        std = std / scale
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Positive intensity (mean peak normalized to 1)")
ax.set_title("Marker-positive intensity vs transverse distance (normalized)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_INTENSITY_NORM_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_INTENSITY_NORM_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote normalized intensity plot:", PLOT_INTENSITY_NORM_PATH.relative_to(ROOT).as_posix())


## Plot (Positive Intensity, Mean Peak Normalized, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0
max_mean = 1.0

scale_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    max_mean = float(np.nanmax(summary_sub["mean_intensity_fraction"].to_numpy(dtype=float)))
    scale_by_marker[marker_key] = max_mean if np.isfinite(max_mean) and max_mean > 0 else np.nan

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    scale = scale_by_marker.get(marker_key, np.nan)
    for file_id, g in sub.groupby("file_id"):
        y = g["positive_intensity_fraction"].to_numpy(dtype=float)
        if np.isfinite(scale) and scale > 0:
            y = y / scale
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_intensity_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    if np.isfinite(scale) and scale > 0:
        mean = mean / scale
        std = std / scale
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Positive intensity (mean peak normalized to 1)")
ax.set_title("Marker-positive intensity vs transverse distance (normalized, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_INTENSITY_NORM_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_INTENSITY_NORM_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote normalized intensity plot (CI):", PLOT_INTENSITY_NORM_CI_PATH.relative_to(ROOT).as_posix())


## Plot (All Intensity, Not Just Positive Mask)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_intensity_all_fraction"].to_numpy(dtype=float)))
max_mean = float(np.nanmax(summary_df["mean_intensity_all_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["intensity_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_intensity_all_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_intensity_all_fraction"].to_numpy(dtype=float)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of all corrected intensity in bin")
ax.set_title("Marker intensity vs transverse distance (all mask pixels)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_INTENSITY_ALL_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_INTENSITY_ALL_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote all-intensity profile plot:", PLOT_INTENSITY_ALL_PATH.relative_to(ROOT).as_posix())


## Plot (All Intensity, Not Just Positive Mask, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_intensity_all_fraction"].to_numpy(dtype=float)))
max_mean = float(np.nanmax(summary_df["mean_intensity_all_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["intensity_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_intensity_all_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_intensity_all_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of all corrected intensity in bin")
ax.set_title("Marker intensity vs transverse distance (all mask pixels, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_INTENSITY_ALL_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_INTENSITY_ALL_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote all-intensity profile plot (CI):", PLOT_INTENSITY_ALL_CI_PATH.relative_to(ROOT).as_posix())


## Plot (Soft-threshold Intensity, Raw)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_soft_intensity_fraction"].to_numpy(dtype=float)))
max_mean = float(np.nanmax(summary_df["mean_soft_intensity_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["soft_intensity_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_soft_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_soft_intensity_fraction"].to_numpy(dtype=float)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of soft-threshold intensity in bin")
ax.set_title("Marker intensity vs transverse distance (soft threshold)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_SOFT_RAW_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_SOFT_RAW_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote soft-threshold profile plot:", PLOT_SOFT_RAW_PATH.relative_to(ROOT).as_posix())


## Plot (Soft-threshold Intensity, Raw, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = float(np.nanmax(summary_df["mean_soft_intensity_fraction"].to_numpy(dtype=float)))
max_mean = float(np.nanmax(summary_df["mean_soft_intensity_fraction"].to_numpy(dtype=float)))

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    for file_id, g in sub.groupby("file_id"):
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            g["soft_intensity_fraction"].to_numpy(dtype=float),
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_soft_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_soft_intensity_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Fraction of soft-threshold intensity in bin")
ax.set_title("Marker intensity vs transverse distance (soft threshold, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_SOFT_RAW_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_SOFT_RAW_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote soft-threshold profile plot (CI):", PLOT_SOFT_RAW_CI_PATH.relative_to(ROOT).as_posix())


## Plot (Soft-threshold Intensity, Mean Peak Normalized)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0
max_mean = 1.0

scale_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    max_mean = float(np.nanmax(summary_sub["mean_soft_intensity_fraction"].to_numpy(dtype=float)))
    scale_by_marker[marker_key] = max_mean if np.isfinite(max_mean) and max_mean > 0 else np.nan

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    scale = scale_by_marker.get(marker_key, np.nan)
    for file_id, g in sub.groupby("file_id"):
        y = g["soft_intensity_fraction"].to_numpy(dtype=float)
        if np.isfinite(scale) and scale > 0:
            y = y / scale
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_soft_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_soft_intensity_fraction"].to_numpy(dtype=float)
    if np.isfinite(scale) and scale > 0:
        mean = mean / scale
        std = std / scale
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - std, mean + std, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Soft-threshold intensity (mean peak normalized to 1)")
ax.set_title("Marker intensity vs transverse distance (soft threshold, normalized)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_SOFT_NORM_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_SOFT_NORM_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote normalized soft-threshold plot:", PLOT_SOFT_NORM_PATH.relative_to(ROOT).as_posix())


## Plot (Soft-threshold Intensity, Mean Peak Normalized, 95% CI)


In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.2), constrained_layout=True)
max_mean = 1.0
max_mean = 1.0

scale_by_marker = {}
for marker_key in MARKERS:
    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    max_mean = float(np.nanmax(summary_sub["mean_soft_intensity_fraction"].to_numpy(dtype=float)))
    scale_by_marker[marker_key] = max_mean if np.isfinite(max_mean) and max_mean > 0 else np.nan

for marker_key in MARKERS:
    color = COLOR_MAP.get(marker_key, "0.4")
    sub = profile_df.loc[profile_df["marker_key"].astype(str) == marker_key]
    scale = scale_by_marker.get(marker_key, np.nan)
    for file_id, g in sub.groupby("file_id"):
        y = g["soft_intensity_fraction"].to_numpy(dtype=float)
        if np.isfinite(scale) and scale > 0:
            y = y / scale
        ax.plot(
            g["bin_center_um"].to_numpy(dtype=float),
            y,
            color=color,
            alpha=0.15,
            linewidth=1.0,
        )

    summary_sub = summary_df.loc[summary_df["marker_key"].astype(str) == marker_key].copy()
    x = summary_sub["bin_center_um"].to_numpy(dtype=float)
    mean = summary_sub["mean_soft_intensity_fraction"].to_numpy(dtype=float)
    std = summary_sub["std_soft_intensity_fraction"].to_numpy(dtype=float)
    n = summary_sub["n_organoids"].to_numpy(dtype=float)
    if np.isfinite(scale) and scale > 0:
        mean = mean / scale
        std = std / scale
    ci = 1.96 * std / np.sqrt(n)
    ax.plot(x, mean, color=color, linewidth=2.4, label=marker_key.upper())
    ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.18, linewidth=0)

ax.set_xlabel("Absolute transverse distance from axis (um)")
ax.set_ylabel("Soft-threshold intensity (mean peak normalized to 1)")
ax.set_title("Marker intensity vs transverse distance (soft threshold, normalized, 95% CI)")
ax.grid(axis="y", alpha=0.25, linewidth=0.6)
ax.legend(frameon=False)

fig.savefig(PLOT_SOFT_NORM_CI_PATH, dpi=180, bbox_inches="tight")
fig.savefig(PLOT_SOFT_NORM_CI_PATH.with_suffix(".svg"), bbox_inches="tight")
plt.close(fig)
print("Wrote normalized soft-threshold plot (CI):", PLOT_SOFT_NORM_CI_PATH.relative_to(ROOT).as_posix())
